In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

pd.set_option('future.no_silent_downcasting', True)

try:
    df = pd.read_excel('asus_laptops.xlsx')
except:
    print("Файл 'asus_laptops.xlsx' не найден!")

df = df.replace('Не указано', np.nan)

def clean_numeric(series):
    return pd.to_numeric(series.astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')

df['ОЗУ'] = clean_numeric(df['ОЗУ'])
df['Объем диска'] = clean_numeric(df['Объем диска'])
df['Частота матрицы'] = clean_numeric(df['Частота матрицы'])

categorical_features = ['Тип', 'Линейка', 'ОС', 'Процессор', 'Разрешение', 'Тип матрицы', 'Видеокарта', 'Тип диска']
numeric_features = ['Дата выхода', 'Ядра', 'Частота CPU', 'Диагональ', 'Частота матрицы', 'ОЗУ', 'Объем диска']

for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=['Цена'])

X = df.drop(columns=['Название', 'Цена'])
y = df['Цена'].values.reshape(-1, 1)

y_scaler = MinMaxScaler()
y_scaled = y_scaler.fit_transform(y)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

X_processed = preprocessor.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_processed, y_scaled, test_size=0.2, random_state=42)

model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='linear')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')
model.fit(X_train, y_train, epochs=500, batch_size=4, verbose=0)

options = {col: sorted(df[col].dropna().unique().tolist()) for col in categorical_features}
ranges = {col: (df[col].min(), df[col].max(), df[col].median()) for col in numeric_features}

display(HTML("""
<style>
    .main-app { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 30px; border-radius: 20px; font-family: sans-serif; color: white; width: 98%; box-shadow: 0 20px 50px rgba(0,0,0,0.3); }
    .section-card { background: rgba(255, 255, 255, 0.15); backdrop-filter: blur(15px); border-radius: 15px; padding: 20px; margin: 10px; border: 1px solid rgba(255, 255, 255, 0.2); }
    .section-title { font-size: 18px; font-weight: 900; text-align: center; margin-bottom: 10px; color: #00f2fe; text-transform: uppercase; }
    .predict-btn { background: #00f2fe !important; border: none !important; color: #1a1a2e !important; font-size: 22px !important; font-weight: 900 !important; border-radius: 12px !important; height: 60px !important; margin-top: 10px !important; cursor: pointer; }
    .price-display { background: white; color: #2d3436; padding: 25px; border-radius: 20px; text-align: center; margin-top: 15px; border: 3px solid #00f2fe; }
    .summary-text { font-size: 16px; margin-bottom: 10px; }
    .final-price-value { color: #2ecc71; font-size: 48px; font-weight: 900; display: block; }
</style>
"""))

groups = {
    "🏷️ Основное": ['Тип', 'Линейка', 'ОС', 'Дата выхода'],
    "🖥️ Экран": ['Диагональ', 'Разрешение', 'Тип матрицы', 'Частота матрицы'],
    "🚀 Процессор": ['Процессор', 'Ядра', 'Частота CPU'],
    "💾 Память": ['Видеокарта', 'ОЗУ', 'Тип диска', 'Объем диска']
}

inputs = {}
for col in numeric_features + categorical_features:
    style = {'description_width': '140px'}
    layout = {'width': '98%', 'margin': '4px 0'}
    if col in categorical_features:
        opts = options[col]
        if col == 'Тип': opts = [o for o in opts if o in ['Ноутбук', 'Ультрабук']]
        inputs[col] = widgets.Dropdown(options=opts, description=col, style=style, layout=layout)
    else:
        inputs[col] = widgets.FloatText(value=ranges[col][2], description=col, style=style, layout=layout)

def create_section(title, keys):
    return widgets.VBox([widgets.HTML(f"<div class='section-title'>{title}</div>"),
                         widgets.VBox([inputs[k] for k in keys])]).add_class("section-card")

row1 = widgets.HBox([create_section("🏷️ Основное", groups["🏷️ Основное"]), create_section("🖥️ Экран", groups["🖥️ Экран"])])
row2 = widgets.HBox([create_section("🚀 Процессор", groups["🚀 Процессор"]), create_section("💾 Память", groups["💾 Память"])])
predict_btn = widgets.Button(description="УЗНАТЬ СТОИМОСТЬ", layout={'width': '100%'}).add_class("predict-btn")
output = widgets.Output()

def on_click(b):
    with output:
        clear_output()
        data = {col: [inputs[col].value] for col in (numeric_features + categorical_features)}
        proc = preprocessor.transform(pd.DataFrame(data))
        price = y_scaler.inverse_transform(model.predict(proc, verbose=0))[0][0]

        display(HTML(f"""
            <div class='price-display'>
                <div class='summary-text'>Конфигурация: {inputs['Процессор'].value}, {inputs['ОЗУ'].value}GB RAM, {inputs['Видеокарта'].value}</div>
                <div style='font-size:18px; font-weight:700;'>Оценочная стоимость:</div>
                <div class='final-price-value'>{int(round(price)): } BYN</div>
            </div>
        """))

predict_btn.on_click(on_click)
header = widgets.HTML("<div style='text-align:center;'><h2 style='color:white;'>Прогнозирование цен ASUS</h2></div>")
display(widgets.VBox([header, row1, row2, predict_btn, output]).add_class("main-app"))